In [1]:
import sys; sys.path.insert(0, "..")
import matplotlib
matplotlib.use("Agg")
import json, os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from datetime import date
from dataclasses import fields as dc_fields
from pid_optimizer.gains import Gains, HERMIT_REFERENCE_GAINS, GAIN_BOUNDS
from pid_optimizer.plant_model import PlantModel
from pid_optimizer.pid_simulator import PX4Simulator
from pid_optimizer.params_io import save_params

BEST_GAINS_PATH = "../artifacts/best_gains.json"
PLANT_PATH      = "../artifacts/plant_model.json"

with open(BEST_GAINS_PATH) as f:
    best = json.load(f)

candidates = best["candidates"]
ref_fitness = best["reference_fitness"]
print(f"Reference fitness: {ref_fitness:.4f}")
print(f"Top candidates:")
for i, c in enumerate(candidates):
    print(f"  #{i+1}  fitness={c['fitness']:.4f}  (\u0394={c['fitness']-ref_fitness:+.4f})")

Reference fitness: 1521.0387
Top candidates:
  #1  fitness=1000000.0000  (Δ=+998478.9613)
  #2  fitness=1000000.0000  (Δ=+998478.9613)
  #3  fitness=1000000.0000  (Δ=+998478.9613)
  #4  fitness=1000000.0000  (Δ=+998478.9613)
  #5  fitness=1000000.0000  (Δ=+998478.9613)


In [2]:
gain_names = [f.name for f in dc_fields(Gains)]
ref_arr    = HERMIT_REFERENCE_GAINS.to_array()
best_arr   = np.array(candidates[0]["gains"])

rows = []
for name, ref_val, opt_val, bounds in zip(gain_names, ref_arr, best_arr, GAIN_BOUNDS):
    pct_change = 100 * (opt_val - ref_val) / (ref_val + 1e-12)
    flag = ">50%" if abs(pct_change) > 50 else ""
    rows.append({"parameter": name, "reference": f"{ref_val:.6f}",
                  "optimized": f"{opt_val:.6f}", "change_%": f"{pct_change:+.1f}", "flag": flag})

df = pd.DataFrame(rows)
df_sorted = pd.concat([df[df["flag"] != ""], df[df["flag"] == ""]])
print(df_sorted.to_string(index=False))

  parameter reference optimized           change_% flag
 rollrate_P  0.215000  0.389238              +81.0 >50%
 rollrate_D  0.002200  0.010303             +368.3 >50%
 rollrate_K  0.450000  2.122367             +371.6 >50%
pitchrate_P  0.215000  0.056147              -73.9 >50%
pitchrate_I  0.175000  0.487811             +178.7 >50%
pitchrate_D  0.002200  0.009134             +315.2 >50%
pitchrate_K  0.450000  2.379586             +428.8 >50%
  yawrate_P  0.154000  0.072776              -52.7 >50%
  yawrate_K  1.750000  2.787618              +59.3 >50%
      yaw_P  5.000000  2.495364              -50.1 >50%
    z_vel_P  4.000000  6.638286              +66.0 >50%
    z_vel_I  2.000000  3.158322              +57.9 >50%
    z_vel_D  0.000000  1.516175 +151617548017074.8 >50%
    z_pos_P  1.000000  1.944326              +94.4 >50%
 rollrate_I  0.175000  0.219439              +25.4     
  yawrate_I  0.160000  0.225193              +40.7     
  yawrate_D  0.003500  0.003708               +5

In [3]:
import pyulog
from scipy.spatial.transform import Rotation as R_

LOG_PATH = "../px4_logs/Hermit/testes_PID_position_31-08/log_12_2024-8-30-16-21-54.ulg"
plant = PlantModel.load(PLANT_PATH)
log = pyulog.ULog(LOG_PATH, disable_str_exceptions=True)
t0 = log.start_timestamp

def to_df(log, topic, multi_id=0):
    d = next(d for d in log.data_list if d.name == topic and d.multi_id == multi_id)
    df = pd.DataFrame({f: d.data[f] for f in d.data})
    df["t_s"] = (df["timestamp"] - t0) / 1e6
    return df.sort_values("t_s").reset_index(drop=True)

DT, T_MAX = 0.004, 20.0
t_sim = np.arange(0, T_MAX, DT)
pos_sp_df = to_df(log, "vehicle_local_position_setpoint")
att_sp_df = to_df(log, "vehicle_attitude_setpoint")
q_sp = att_sp_df[["q_d[0]","q_d[1]","q_d[2]","q_d[3]"]].values
yaw_sp = R_.from_quat(q_sp[:, [1,2,3,0]]).as_euler("xyz")[:, 2]

def interp_col(src_df, col, t_out):
    mask = ~src_df[col].isna()
    if mask.sum() < 2:
        return np.zeros(len(t_out))
    return np.interp(t_out, src_df["t_s"][mask].values, src_df[col][mask].values)

setpoints = pd.DataFrame({
    "x": interp_col(pos_sp_df, "x", t_sim), "y": interp_col(pos_sp_df, "y", t_sim),
    "z": interp_col(pos_sp_df, "z", t_sim), "vx": interp_col(pos_sp_df, "vx", t_sim),
    "vy": interp_col(pos_sp_df, "vy", t_sim), "vz": interp_col(pos_sp_df, "vz", t_sim),
    "roll": float('nan'), "pitch": float('nan'),
    "yaw": np.interp(t_sim, att_sp_df["t_s"].values, yaw_sp),
})

ref_gains = HERMIT_REFERENCE_GAINS
opt_gains = Gains.from_array(np.array(candidates[0]["gains"]))

res_ref = PX4Simulator(plant, ref_gains).run(setpoints, dt=DT)
res_opt = PX4Simulator(plant, opt_gains).run(setpoints, dt=DT)

fig, axes = plt.subplots(3, 1, figsize=(14, 8), sharex=True)
for i, (col, label) in enumerate([("x","X(m)"),("y","Y(m)"),("z","Z(m)")]):
    axes[i].plot(t_sim[:len(res_ref)], res_ref[col].values, label="reference gains", alpha=0.8)
    axes[i].plot(t_sim[:len(res_opt)], res_opt[col].values, label="optimized gains", alpha=0.8)
    axes[i].plot(t_sim, setpoints[col].values, "k--", label="setpoint", alpha=0.4)
    axes[i].set_ylabel(label); axes[i].legend()
axes[-1].set_xlabel("Time (s)")
plt.suptitle("Simulated trajectory: reference vs optimized gains")
plt.tight_layout(); plt.savefig("/tmp/ref_vs_opt.png"); plt.close()
print("Saved comparison plot to /tmp/ref_vs_opt.png")

Saved comparison plot to /tmp/ref_vs_opt.png


In [4]:
CANDIDATE_IDX = 0  # Change to pick a different candidate
chosen_gains = Gains.from_array(np.array(candidates[CANDIDATE_IDX]["gains"]))
chosen_fitness = candidates[CANDIDATE_IDX]["fitness"]

today = date.today().isoformat()
out_path = f"../artifacts/optimized_hermit_{today}.params"

# save_params(gains: Gains, path: str) — writes our 2-column format
save_params(chosen_gains, out_path)

print(f"Exported to: {out_path}")
print(f"Fitness: {chosen_fitness:.4f} (reference: {ref_fitness:.4f})")
print(f"Change: {100*(chosen_fitness-ref_fitness)/ref_fitness:+.1f}%")

# Show first few lines
with open(out_path) as f:
    for line in f.readlines()[:5]:
        print(repr(line))

Exported to: ../artifacts/optimized_hermit_2026-05-14.params
Fitness: 1000000.0000 (reference: 1521.0387)
Change: +65644.5%
'# Exported by pid_optimizer\n'
'MC_PITCHRATE_D\t0.00913368\n'
'MC_PITCHRATE_I\t0.487811\n'
'MC_PITCHRATE_K\t2.37959\n'
'MC_PITCHRATE_P\t0.0561469\n'
